In [15]:
# ================================================================
# HOTEL BOOKING CANCELLATION PREDICTION
# XGBOOST + LIGHTGBM
# FEATURE ENGINEERING + NO ONE-HOT ENCODING
# END-TO-END PIPELINE
# ================================================================

import pandas as pd
import numpy as np
import json
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ================================================================
# 1. PATHS
# ================================================================

print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

# Notebook is inside /models
DATA_PATH = Path("../master_hotel_bookings_cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}\n"
        "Make sure master_hotel_bookings_cleaned.csv is in the project root."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)


# ================================================================
# 2. BASIC CLEANING
# ================================================================

print("\n" + "=" * 70)
print("BASIC CLEANING")
print("=" * 70)

# Remove leakage columns
leakage_columns = [
    "reservation_status",
    "reservation_status_date"
]

leakage_columns = [
    col for col in leakage_columns
    if col in df.columns
]

df = df.drop(columns=leakage_columns)

print("\nLeakage columns removed:")
print(leakage_columns)


# ================================================================
# 3. TARGET
# ================================================================

TARGET = "is_canceled"

if TARGET not in df.columns:
    raise ValueError("Target column 'is_canceled' not found.")

# Remove rows where target is missing
df = df.dropna(subset=[TARGET]).copy()

df[TARGET] = df[TARGET].astype(int)

print("\nTarget distribution:")
print(df[TARGET].value_counts())

print("\nTarget percentage:")
print(
    (df[TARGET].value_counts(normalize=True) * 100).round(2)
)


# ================================================================
# 4. FEATURE ENGINEERING
# ================================================================

print("\n" + "=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)


# ------------------------------------------------
# 4.1 Domestic / International
# ------------------------------------------------

if "country" in df.columns:

    df["is_domestic"] = (
        df["country"]
        .fillna("Unknown")
        .astype(str)
        .eq("PRT")
        .astype(int)
    )

else:
    df["is_domestic"] = 0


# ------------------------------------------------
# 4.2 Traveler Type
# ------------------------------------------------

def create_traveler_type(row):

    adults = row["adults"] if pd.notna(row["adults"]) else 0
    children = row["children"] if pd.notna(row["children"]) else 0
    babies = row["babies"] if pd.notna(row["babies"]) else 0

    total_people = adults + children + babies

    if children > 0 or babies > 0:
        return "Family"

    elif adults == 1:
        return "Solo"

    elif adults == 2:
        return "Couple"

    elif adults > 2:
        return "Group"

    elif total_people == 0:
        return "Unknown"

    return "Other"


df["traveler_type"] = df.apply(
    create_traveler_type,
    axis=1
)


# ------------------------------------------------
# 4.3 Parking requirement
# ------------------------------------------------

if "required_car_parking_spaces" in df.columns:

    df["requires_parking"] = (
        df["required_car_parking_spaces"] > 0
    ).astype(int)

else:
    df["requires_parking"] = 0


# ------------------------------------------------
# 4.4 Clean ADR
# ------------------------------------------------

if "adr" in df.columns:

    median_adr = df.loc[
        df["adr"] >= 0,
        "adr"
    ].median()

    df["adr_clean"] = df["adr"].where(
        df["adr"] >= 0,
        median_adr
    )

else:
    df["adr_clean"] = 0


# ------------------------------------------------
# 4.5 Log ADR
# ------------------------------------------------

df["adr_log"] = np.log1p(
    df["adr_clean"].clip(lower=0)
)


# ------------------------------------------------
# 4.6 Total stay nights
# ------------------------------------------------

df["total_stay_nights"] = (
    df["stays_in_weekend_nights"].fillna(0)
    +
    df["stays_in_week_nights"].fillna(0)
)


# ------------------------------------------------
# 4.7 Room changed
# ------------------------------------------------

if (
    "reserved_room_type" in df.columns
    and "assigned_room_type" in df.columns
):

    df["room_changed"] = (
        df["reserved_room_type"].astype(str)
        != df["assigned_room_type"].astype(str)
    ).astype(int)

else:
    df["room_changed"] = 0


# ------------------------------------------------
# 4.8 Booking changes
# ------------------------------------------------

if "booking_changes" in df.columns:

    df["has_booking_changes"] = (
        df["booking_changes"] > 0
    ).astype(int)

else:
    df["has_booking_changes"] = 0


# ------------------------------------------------
# 4.9 Booking source type
# ------------------------------------------------
# Already present in the cleaned dataset.
# If absent, create a fallback.

if "booking_source_type" not in df.columns:

    if "market_segment" in df.columns:

        df["booking_source_type"] = (
            df["market_segment"]
            .fillna("Unknown")
            .astype(str)
        )

    else:

        df["booking_source_type"] = "Unknown"


# ------------------------------------------------
# 4.10 Has babies
# ------------------------------------------------

if "babies" in df.columns:

    df["has_babies"] = (
        df["babies"].fillna(0) > 0
    ).astype(int)

else:
    df["has_babies"] = 0


# ------------------------------------------------
# 4.11 Is family
# ------------------------------------------------

if "children" in df.columns or "babies" in df.columns:

    children = (
        df["children"].fillna(0)
        if "children" in df.columns
        else 0
    )

    babies = (
        df["babies"].fillna(0)
        if "babies" in df.columns
        else 0
    )

    df["is_family"] = (
        (children + babies) > 0
    ).astype(int)

else:
    df["is_family"] = 0


print("\nEngineered features:")
print("""
- is_domestic
- traveler_type
- requires_parking
- adr_clean
- adr_log
- total_stay_nights
- room_changed
- has_booking_changes
- booking_source_type
- has_babies
- is_family
""")


# ================================================================
# 5. REMOVE TARGET + HIGH-MISSING ID-LIKE FEATURES
# ================================================================

columns_to_remove = [
    TARGET,
    "agent",
    "company"
]

columns_to_remove = [
    col for col in columns_to_remove
    if col in df.columns
]

X = df.drop(columns=columns_to_remove).copy()
y = df[TARGET].copy()

print("Columns removed before modelling:")
print(columns_to_remove)


# ================================================================
# 6. REMOVE OLD ENCODED COLUMNS
# ================================================================

old_encoded_columns = [
    "customer_type_Group",
    "customer_type_Transient",
    "customer_type_Transient-Party"
]

old_encoded_columns = [
    col for col in old_encoded_columns
    if col in X.columns
]

X = X.drop(columns=old_encoded_columns)

print("\nOld encoded columns removed:")
print(old_encoded_columns)

print("\nFinal X shape:", X.shape)


# ================================================================
# 7. IDENTIFY CATEGORICAL + NUMERIC FEATURES
# ================================================================

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of categorical features:",
      len(categorical_features))

print("\nNumber of numeric features:",
      len(numeric_features))


# ================================================================
# 8. HANDLE MISSING VALUES
# ================================================================

# Categorical → Unknown
for col in categorical_features:

    X[col] = (
        X[col]
        .fillna("Unknown")
        .astype(str)
    )


# Numeric → median
for col in numeric_features:

    X[col] = X[col].fillna(
        X[col].median()
    )


print("\nRemaining missing values:")
print(X.isna().sum().sum())


# ================================================================
# 9. CONVERT CATEGORICAL COLUMNS TO PANDAS CATEGORY
# ================================================================
# IMPORTANT:
# No One-Hot Encoding
# No Label Encoding
# Native categorical handling is used.

for col in categorical_features:

    X[col] = X[col].astype("category")


# ================================================================
# 10. TRAIN / VALIDATION / TEST SPLIT
# ================================================================

print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)

# First: 80% train+validation, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Then split 80% into:
# 70% total training
# 10% validation
#
# 0.125 of 80% = 10% of total

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.125,
    stratify=y_temp,
    random_state=42
)

print("Training   :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)


# ================================================================
# 11. SCALE POSITIVE WEIGHT
# ================================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = (
    negative_count / positive_count
)

print(
    "\nScale Pos Weight:",
    round(scale_pos_weight, 4)
)


# ================================================================
# 12. XGBOOST
# ================================================================

print("\n" + "=" * 70)
print("TRAINING XGBOOST")
print("=" * 70)

xgb_model = XGBClassifier(

    n_estimators=500,

    max_depth=8,

    learning_rate=0.1,

    subsample=0.8,

    colsample_bytree=1.0,

    objective="binary:logistic",

    eval_metric="logloss",

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    tree_method="hist",

    enable_categorical=True
)


xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost training completed!")


# ================================================================
# 13. XGBOOST VALIDATION THRESHOLD
# ================================================================

xgb_val_prob = xgb_model.predict_proba(
    X_val
)[:, 1]


thresholds = np.arange(
    0.30,
    0.71,
    0.01
)

best_xgb_threshold = 0.50
best_xgb_val_f1 = -1


for threshold in thresholds:

    val_pred = (
        xgb_val_prob >= threshold
    ).astype(int)

    score = f1_score(
        y_val,
        val_pred
    )

    if score > best_xgb_val_f1:

        best_xgb_val_f1 = score
        best_xgb_threshold = threshold


# ================================================================
# 14. XGBOOST TEST RESULTS
# ================================================================

xgb_test_prob = xgb_model.predict_proba(
    X_test
)[:, 1]

xgb_test_pred = (
    xgb_test_prob >= best_xgb_threshold
).astype(int)


xgb_accuracy = accuracy_score(
    y_test,
    xgb_test_pred
)

xgb_precision = precision_score(
    y_test,
    xgb_test_pred,
    zero_division=0
)

xgb_recall = recall_score(
    y_test,
    xgb_test_pred,
    zero_division=0
)

xgb_f1 = f1_score(
    y_test,
    xgb_test_pred,
    zero_division=0
)

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_test_prob
)


print("\n" + "-" * 60)
print(
    f"XGBoost - Tuned Threshold "
    f"{best_xgb_threshold:.2f}"
)
print("-" * 60)

print(
    f"Accuracy : {xgb_accuracy:.4f}"
)

print(
    f"Precision: {xgb_precision:.4f}"
)

print(
    f"Recall   : {xgb_recall:.4f}"
)

print(
    f"F1 Score : {xgb_f1:.4f}"
)

print(
    f"ROC-AUC  : {xgb_roc_auc:.4f}"
)

print(
    "\nBest XGBoost validation threshold:",
    round(best_xgb_threshold, 2)
)

print(
    "Best XGBoost validation F1:",
    round(best_xgb_val_f1, 4)
)


# ================================================================
# 15. XGBOOST FEATURE IMPORTANCE
# ================================================================

xgb_feature_importance = pd.DataFrame({

    "feature": X_train.columns,

    "importance": xgb_model.feature_importances_

})

xgb_feature_importance = (
    xgb_feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("TOP 20 XGBOOST FEATURES")
print("=" * 70)

print(
    xgb_feature_importance.head(20)
    .to_string(index=False)
)


# ================================================================
# 16. LIGHTGBM
# ================================================================

print("\n" + "=" * 70)
print("TRAINING LIGHTGBM")
print("=" * 70)


lgb_model = LGBMClassifier(

    n_estimators=500,

    learning_rate=0.1,

    max_depth=8,

    subsample=0.8,

    colsample_bytree=1.0,

    objective="binary",

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    verbosity=-1
)


lgb_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

print("LightGBM training completed!")


# ================================================================
# 17. LIGHTGBM VALIDATION THRESHOLD
# ================================================================

lgb_val_prob = lgb_model.predict_proba(
    X_val
)[:, 1]


best_lgb_threshold = 0.50
best_lgb_val_f1 = -1


for threshold in thresholds:

    val_pred = (
        lgb_val_prob >= threshold
    ).astype(int)

    score = f1_score(
        y_val,
        val_pred
    )

    if score > best_lgb_val_f1:

        best_lgb_val_f1 = score
        best_lgb_threshold = threshold


# ================================================================
# 18. LIGHTGBM TEST RESULTS
# ================================================================

lgb_test_prob = lgb_model.predict_proba(
    X_test
)[:, 1]

lgb_test_pred = (
    lgb_test_prob >= best_lgb_threshold
).astype(int)


lgb_accuracy = accuracy_score(
    y_test,
    lgb_test_pred
)

lgb_precision = precision_score(
    y_test,
    lgb_test_pred,
    zero_division=0
)

lgb_recall = recall_score(
    y_test,
    lgb_test_pred,
    zero_division=0
)

lgb_f1 = f1_score(
    y_test,
    lgb_test_pred,
    zero_division=0
)

lgb_roc_auc = roc_auc_score(
    y_test,
    lgb_test_prob
)


print("\n" + "-" * 60)
print(
    f"LightGBM - Tuned Threshold "
    f"{best_lgb_threshold:.2f}"
)
print("-" * 60)

print(
    f"Accuracy : {lgb_accuracy:.4f}"
)

print(
    f"Precision: {lgb_precision:.4f}"
)

print(
    f"Recall   : {lgb_recall:.4f}"
)

print(
    f"F1 Score : {lgb_f1:.4f}"
)

print(
    f"ROC-AUC  : {lgb_roc_auc:.4f}"
)

print(
    "\nBest LightGBM validation threshold:",
    round(best_lgb_threshold, 2)
)

print(
    "Best LightGBM validation F1:",
    round(best_lgb_val_f1, 4)
)


# ================================================================
# 19. LIGHTGBM FEATURE IMPORTANCE
# ================================================================

lgb_feature_importance = pd.DataFrame({

    "feature": X_train.columns,

    "importance": lgb_model.feature_importances_

})

lgb_feature_importance = (
    lgb_feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("TOP 20 LIGHTGBM FEATURES")
print("=" * 70)

print(
    lgb_feature_importance.head(20)
    .to_string(index=False)
)


# ================================================================
# 20. FINAL MODEL COMPARISON
# ================================================================

print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)


comparison = pd.DataFrame({

    "Model": [
        "XGBoost",
        "LightGBM"
    ],

    "Threshold": [
        best_xgb_threshold,
        best_lgb_threshold
    ],

    "Accuracy": [
        xgb_accuracy,
        lgb_accuracy
    ],

    "Precision": [
        xgb_precision,
        lgb_precision
    ],

    "Recall": [
        xgb_recall,
        lgb_recall
    ],

    "F1": [
        xgb_f1,
        lgb_f1
    ],

    "ROC_AUC": [
        xgb_roc_auc,
        lgb_roc_auc
    ]
})


print(
    comparison.round(4).to_string(index=False)
)


# ================================================================
# 21. CONFUSION MATRICES
# ================================================================

print("\n" + "=" * 70)
print("XGBOOST CONFUSION MATRIX")
print("=" * 70)

print(
    confusion_matrix(
        y_test,
        xgb_test_pred
    )
)


print("\n" + "=" * 70)
print("LIGHTGBM CONFUSION MATRIX")
print("=" * 70)

print(
    confusion_matrix(
        y_test,
        lgb_test_pred
    )
)


# ================================================================
# 22. SAVE MODELS
# ================================================================

print("\n" + "=" * 70)
print("SAVING FINAL MODELS")
print("=" * 70)


OUTPUT_DIR = Path(".")

# XGBoost
joblib.dump(
    xgb_model,
    OUTPUT_DIR / "xgboost_no_encoding_model.pkl"
)

# LightGBM
joblib.dump(
    lgb_model,
    OUTPUT_DIR / "lightgbm_no_encoding_model.pkl"
)


# ================================================================
# 23. SAVE FEATURE IMPORTANCE
# ================================================================

xgb_feature_importance.to_csv(
    OUTPUT_DIR / "xgboost_feature_importance.csv",
    index=False
)

lgb_feature_importance.to_csv(
    OUTPUT_DIR / "lightgbm_feature_importance.csv",
    index=False
)


# ================================================================
# 24. SAVE MODEL CONFIGURATION
# ================================================================

model_config = {

    "target": TARGET,

    "dataset_shape": list(df.shape),

    "final_feature_count": X.shape[1],

    "categorical_feature_count":
        len(categorical_features),

    "numeric_feature_count":
        len(numeric_features),

    "categorical_features":
        categorical_features,

    "numeric_features":
        numeric_features,

    "removed_leakage_columns":
        leakage_columns,

    "removed_columns":
        columns_to_remove,

    "old_encoded_columns_removed":
        old_encoded_columns,

    "train_shape":
        list(X_train.shape),

    "validation_shape":
        list(X_val.shape),

    "test_shape":
        list(X_test.shape),

    "scale_pos_weight":
        float(scale_pos_weight),

    "xgboost_threshold":
        float(best_xgb_threshold),

    "lightgbm_threshold":
        float(best_lgb_threshold),

    "xgboost_metrics": {

        "accuracy":
            float(xgb_accuracy),

        "precision":
            float(xgb_precision),

        "recall":
            float(xgb_recall),

        "f1":
            float(xgb_f1),

        "roc_auc":
            float(xgb_roc_auc)
    },

    "lightgbm_metrics": {

        "accuracy":
            float(lgb_accuracy),

        "precision":
            float(lgb_precision),

        "recall":
            float(lgb_recall),

        "f1":
            float(lgb_f1),

        "roc_auc":
            float(lgb_roc_auc)
    }
}


with open(
    OUTPUT_DIR / "model_config.json",
    "w"
) as f:

    json.dump(
        model_config,
        f,
        indent=4
    )


# ================================================================
# 25. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("COMPLETE")
print("=" * 70)

print(
    "\nFinal feature count:",
    X.shape[1]
)

print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Numeric features:",
    len(numeric_features)
)

print("\nFiles created:")

print("1. xgboost_no_encoding_model.pkl")
print("2. lightgbm_no_encoding_model.pkl")
print("3. xgboost_feature_importance.csv")
print("4. lightgbm_feature_importance.csv")
print("5. model_config.json")


print("\nTop XGBoost feature:")
print(
    xgb_feature_importance.iloc[0]
)

print("\nTop LightGBM feature:")
print(
    lgb_feature_importance.iloc[0]
)

print("\n" + "=" * 70)
print("PIPELINE FINISHED SUCCESSFULLY")
print("=" * 70)

LOADING DATASET
Dataset loaded successfully!
Original shape: (87396, 47)

BASIC CLEANING

Leakage columns removed:
['reservation_status', 'reservation_status_date']

Target distribution:
is_canceled
0    63371
1    24025
Name: count, dtype: int64

Target percentage:
is_canceled
0    72.51
1    27.49
Name: proportion, dtype: float64

FEATURE ENGINEERING

Engineered features:

- is_domestic
- traveler_type
- requires_parking
- adr_clean
- adr_log
- total_stay_nights
- room_changed
- has_booking_changes
- booking_source_type
- has_babies
- is_family

Columns removed before modelling:
['is_canceled', 'agent', 'company']

Old encoded columns removed:
['customer_type_Group', 'customer_type_Transient', 'customer_type_Transient-Party']

Final X shape: (87396, 44)

Categorical features:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'booking_source_type', 'waiting_list_grou